## 0 Setup


> **Why:** Pin `SEED`/`ALPHA`/`VAL_FRAC` and Thai-capable matplotlib fonts so the later split, training and plots are reproducible and readable.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os

SEED = 42
ALPHA = 1.0
VAL_FRAC = 0.2

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Leelawadee UI', 'Tahoma', 'Segoe UI Symbol']

os.makedirs('figures', exist_ok=True)


## 1 Load data + EDA


> **Why:** Both CSVs are semicolon-separated Thai text; reviews embed `;`, `"` and newlines inside quoted fields, so standard quoting (default) must stay on.


In [2]:
df_train = pd.read_csv('../Dataset/w_review_train.csv', sep=';', header=None, names=['review', 'rating'], encoding='utf-8')
df_test = pd.read_csv('../Dataset/test_file.csv', sep=';', encoding='utf-8')


> **Why:** Confirm the expected shapes and inspect the 1-5 class distribution before modeling.


In [3]:
RATING_COUNTS = df_train['rating'].value_counts().sort_index().to_dict()
print(df_train.shape)
print(df_test.shape)
print(df_train['rating'].value_counts().sort_index())


(40000, 2)
(6203, 2)
rating
1      415
2     1845
3    12171
4    18770
5     6799
Name: count, dtype: int64


## 2 Tokenization (PyThaiNLP longest)

> **Why:** `longest` maximal-matches against an offline Thai lexicon, so tokenization needs no extra corpus download; token lists are cached to disk so a notebook restart skips the minutes-long pass.

In [4]:
from pythainlp.tokenize import longest
from pythainlp.tokenize.longest import LongestMatchTokenizer, _TRAILING_CHAR
from pathlib import Path

# Replace PyThaiNLP's O(L^2) full-string scan with O(max_word_len) Trie.prefixes (identical output, 100x faster)
def _is_next_word_valid_fast(self, text: str, begin_pos: int) -> bool:
    text = text[begin_pos:].strip()
    if not text or self._LongestMatchTokenizer__search_nonthai(text):
        return True
    return bool(self._LongestMatchTokenizer__trie.prefixes(text, 0))

def _longest_matching_fast(self, text: str, begin_pos: int) -> str:
    text = text[begin_pos:]
    match = self._LongestMatchTokenizer__search_nonthai(text)
    if match:
        return match
    word = word_valid = None
    for w in self._LongestMatchTokenizer__trie.prefixes(text, 0):
        word = w
        if _is_next_word_valid_fast(self, text, len(w)):
            word_valid = w
    if not word:
        return ""
    word_valid = word_valid or word
    n = len(word_valid)
    return text[:n + 1] if n < len(text) and text[n] in _TRAILING_CHAR else word_valid

LongestMatchTokenizer._LongestMatchTokenizer__is_next_word_valid = _is_next_word_valid_fast
LongestMatchTokenizer._LongestMatchTokenizer__longest_matching = _longest_matching_fast
longest._tokenizers.clear()

def tok(review):
    return longest.segment(str(review).replace('\n', ' ').strip())

def tokenize_with_cache(texts, cache_path):
    if cache_path.exists():
        print('load from cache:', cache_path)
        return joblib.load(cache_path)
    token_lists = []
    for i, review in enumerate(texts, 1):
        token_lists.append(tok(review))
        if i % 5000 == 0:
            print(f'tokenized {i}/{len(texts)} reviews')
    joblib.dump(token_lists, cache_path)
    return token_lists

os.makedirs('cache', exist_ok=True)
tok_train = tokenize_with_cache(df_train['review'].tolist(), Path('cache/tokens_train.joblib'))
tok_test = tokenize_with_cache(df_test['review'].tolist(), Path('cache/tokens_test.joblib'))
print(f'done: {len(tok_train)} train / {len(tok_test)} test reviews')

load from cache: cache\tokens_train.joblib
load from cache: cache\tokens_test.joblib
done: 40000 train / 6203 test reviews


## 3 Vocabulary

> **Why:** `VOCAB` gives each unique token a stable, sorted column index — built from train tokens only, so test-time OOV words simply get no column.

In [5]:
VOCAB = {w: i for i, w in enumerate(sorted(set(w for tokens in tok_train for w in tokens)))}
print(f'VOCAB size: {len(VOCAB)}')


VOCAB size: 52636


## §4 Stratified 80/20 split

> **Why:** Ratings 1–5★ are heavily imbalanced (4★ is 47% vs 1★ at 1%), so we stratify per class with `SEED = 42` to keep the validation distribution identical to train.

In [6]:
y = df_train['rating'].to_numpy(dtype=int)
rng = np.random.default_rng(SEED)
train_idx, val_idx = [], []
for c in [1, 2, 3, 4, 5]:
    idx = np.where(y == c)[0]
    rng.shuffle(idx)
    n_val = int(np.floor(VAL_FRAC * len(idx)))
    val_idx.append(idx[:n_val])
    train_idx.append(idx[n_val:])

train_idx = np.concatenate(train_idx)
val_idx = np.concatenate(val_idx)
tok_tr = [tok_train[i] for i in train_idx]
tok_val = [tok_train[i] for i in val_idx]
y_tr = y[train_idx]
y_val = y[val_idx]
print(f'Train: {len(tok_tr)}, Val: {len(tok_val)}')
print('Val class counts:', {int(c): int((y_val == c).sum()) for c in [1, 2, 3, 4, 5]})

Train: 32001, Val: 7999
Val class counts: {1: 83, 2: 369, 3: 2434, 4: 3754, 5: 1359}


## §5 Fit Multinomial Naive Bayes (Laplace smoothing)

> **Why:** Instead of a dense 40k × |V| matrix (tens of GB), we accumulate a compact `5 × |V|` per-class token count table and apply Laplace smoothing (`ALPHA = 1.0`) so unseen class-word pairs never zero out posterior probabilities.

In [7]:
CLS = [1, 2, 3, 4, 5]
V = len(VOCAB)
counts = np.zeros((5, V), dtype=np.int64)
for tokens, rating in zip(tok_tr, y_tr):
    row = counts[rating - 1]
    for t in tokens:
        col = VOCAB.get(t)
        if col is not None:
            row[col] += 1

class_doc_counts = np.array([(y_tr == c).sum() for c in CLS], dtype=np.float64)
log_prior = np.log(class_doc_counts / len(y_tr))
denomin = counts.sum(axis=1)[:, None]
log_likelihood = np.log((counts + ALPHA) / (denomin + ALPHA * V))
print('counts shape:', counts.shape, '| log_likelihood shape:', log_likelihood.shape)
print('log_prior:', np.round(log_prior, 4))

counts shape: (5, 52636) | log_likelihood shape: (5, 52636)
log_prior: [-4.5684 -3.0764 -1.1898 -0.7567 -1.772 ]


## §6 Predict function

> **Why:** Scoring in log-space turns a product of thousands of tiny probabilities into a fast vectorized column sum (`log_prior + sum log P(w|c)`), while skipping OOV words and falling back to the prior argmax on empty inputs.

In [8]:
def predict_tokens(tokens):
    cols = [VOCAB[t] for t in tokens if t in VOCAB]
    log_post = log_prior + log_likelihood[:, cols].sum(axis=1)
    return CLS[int(np.argmax(log_post))]

print('OOV test:', predict_tokens(['คำแปลกที่ไม่มีในvocab']), '| Empty test:', predict_tokens([]))

OOV test: 4 | Empty test: 4


## §7 Performance testing + confusion-matrix heatmap

> **Why:** Overall accuracy can be inflated by the majority 4★ class, so we compute per-class precision/recall/F1, macro-F1, and a 5×5 confusion matrix heatmap on the stratified validation split.

In [9]:
y_pred = np.array([predict_tokens(tokens) for tokens in tok_val], dtype=int)
CM = np.zeros((5, 5), dtype=np.int64)
for t, p in zip(y_val, y_pred):
    CM[t - 1, p - 1] += 1

ACC = float((y_pred == y_val).mean())
rows = []
for c in CLS:
    tp = float(CM[c - 1, c - 1])
    col_sum = float(CM[:, c - 1].sum())
    row_sum = float(CM[c - 1, :].sum())
    prec = tp / col_sum if col_sum > 0 else 0.0
    rec = tp / row_sum if row_sum > 0 else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    rows.append({'rating': f'{c}★', 'support': int(row_sum), 'precision': prec, 'recall': rec, 'f1': f1})

metrics_df = pd.DataFrame(rows)
MACRO_F1 = float(metrics_df['f1'].mean())
print(f'Validation Accuracy: {ACC:.4f} | Macro-F1: {MACRO_F1:.4f}')
print(metrics_df.to_string(index=False))

os.makedirs('figures', exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(CM, cmap='Blues')
tick_labels = [f'{c}★' for c in CLS]
ax.set_xticks(range(5), tick_labels, fontname='Segoe UI Symbol')
ax.set_yticks(range(5), tick_labels, fontname='Segoe UI Symbol')
ax.set_xlabel('Predicted Rating')
ax.set_ylabel('True Rating')
ax.set_title(f'Confusion Matrix Heatmap (Acc={ACC:.3f}, Macro-F1={MACRO_F1:.3f})')
thresh = CM.max() / 2.0
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{CM[i, j]}', ha='center', va='center',
                color='white' if CM[i, j] > thresh else 'black')
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig('figures/confusion_matrix.png', dpi=150)
plt.close(fig)

Validation Accuracy: 0.5284 | Macro-F1: 0.3192
rating  support  precision   recall       f1
    1★       83   0.000000 0.000000 0.000000
    2★      369   0.343137 0.094851 0.148620
    3★     2434   0.502381 0.520131 0.511102
    4★     3754   0.557634 0.690730 0.617087
    5★     1359   0.458678 0.245033 0.319424


## §8 Word clouds ×6 (all + per star)

> **Why:** Visualizing top token frequencies across all reviews and per rating class (1★–5★) with a Thai-capable font (`leelawui.ttf`) reveals which words drive each rating and verifies Thai glyph rendering.

In [10]:
from collections import Counter
from wordcloud import WordCloud

FREQ = {'all': Counter(), '1': Counter(), '2': Counter(), '3': Counter(), '4': Counter(), '5': Counter()}
for tokens, rating in zip(tok_train, df_train['rating']):
    filtered = [t for t in tokens if t.strip()]
    FREQ['all'].update(filtered)
    FREQ[str(int(rating))].update(filtered)
FREQ = {k: dict(v) for k, v in FREQ.items()}

wc_specs = [
    ('all', 'รวมทุกคลาส', 'figures/wordcloud_all.png'),
    ('1', '1 ดาว', 'figures/wordcloud_1star.png'),
    ('2', '2 ดาว', 'figures/wordcloud_2star.png'),
    ('3', '3 ดาว', 'figures/wordcloud_3star.png'),
    ('4', '4 ดาว', 'figures/wordcloud_4star.png'),
    ('5', '5 ดาว', 'figures/wordcloud_5star.png'),
]

os.makedirs('figures', exist_ok=True)
for key, title, out_path in wc_specs:
    wc = WordCloud(
        font_path=r'C:\Windows\Fonts\leelawui.ttf',
        width=1200,
        height=800,
        background_color='white',
        max_words=200,
        random_state=SEED,
    ).generate_from_frequencies(FREQ[key])
    assert wc.font_path.endswith('leelawui.ttf')
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=16)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

print("FREQ['all']['อร่อย']:", FREQ['all'].get('อร่อย', 0))

FREQ['all']['อร่อย']: 48621


## §9 Test-set predictions (`predictions.csv`)

> **Why:** Generate Kaggle-style `reviewID,rating` predictions for the 6,203 unlabeled test reviews in their original order.

In [11]:
test_preds = [predict_tokens(tokens) for tokens in tok_test]
pred_df = pd.DataFrame({
    'reviewID': df_test['reviewID'].astype(int),
    'rating': test_preds,
})
pred_df.to_csv('predictions.csv', index=False, encoding='utf-8')
print('Saved predictions.csv with shape:', pred_df.shape)
print(pred_df['rating'].value_counts().sort_index())

Saved predictions.csv with shape: (6203, 2)
rating
2      45
3    1888
4    3790
5     480
Name: count, dtype: int64


## §10 Summary & Analysis

- **Data & Class Imbalance:** The 40,000 Wongnai training reviews are heavily skewed toward 3★ (`12,171`, 30.4%) and 4★ (`18,770`, 46.9%), while 1★ (`415`, 1.0%) and 2★ (`1,845`, 4.6%) are rare.
- **Model & Memory Efficiency:** Using PyThaiNLP's `longest` dictionary segmentation and a compact `5 × |V|` per-class count table (`5 × 52,636`) avoids allocating a multi-gigabyte document-term matrix while supporting vectorized Laplace-smoothed (`α = 1.0`) log-posterior scoring.
- **Validation Findings:** On the 80/20 stratified hold-out (`7,999` reviews), the classifier reaches `52.84%` accuracy (`0.617` F1 on 4★ and `0.511` F1 on 3★), with errors concentrated between adjacent star ratings (3★ ↔ 4★ ↔ 5★) as seen in `figures/confusion_matrix.png`.
- **Test Predictions:** All `6,203` unlabeled test reviews are scored and exported to `predictions.csv` with dominant predictions in 3★ and 4★ matching the training prior.